# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oumaklaus/ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row = one pseudonymized content item (page), across all 32 clients.

All metrics are aggregated over a trailing 90 day window ending at export time. The 30 day comparison sub windows (`last_30d` / `prev_30d`) sit inside that 90 day span and only feed the label; they are not separate time windows.

Expected: 30,000 rows, one per unique `content_id`.

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("https://github.com/oumaklaus/ml-internship/raw/main/data/raw/content_refresh_anonymized.csv")
print(f"rows x cols       : {df.shape[0]:,} x {df.shape[1]}")
print(f"unique content_id : {df['content_id'].nunique():,}")
print(f"unique client_id  : {df['client_id'].nunique()}")

# Grain check: zero duplicates means one row per page.
dupes = df["content_id"].duplicated().sum()
print(f"duplicate content_id rows : {dupes}   (0 = grain holds)")
assert dupes == 0, "grain broken: content_id should be unique"


rows x cols       : 30,000 x 44
unique content_id : 30,000
unique client_id  : 32
duplicate content_id rows : 0   (0 = grain holds)


## 2. Fields: feature / label / context / excluded

**Label / proxy**

| Column | Role | Note |
|---|---|---|
| `is_declining_label` | target | 1 when `trend_direction == "down"` (54.2% positive rate) |
| `trend_direction` | label source, never a feature | the rule that defines the label |
| `trend_pct` | label source, never a feature | the number `trend_direction` is bucketed from |

**Context (grouping and splits only, never model inputs)**

| Column | Role |
|---|---|
| `content_id` | pseudonymous page id, joins and grain checks only |
| `client_id` | pseudonymous client id, use for client holdout splits only |

**Excluded (with reason)**

| Column | Why excluded |
|---|---|
| `impressions_last_30d`, `impressions_prev_30d` | direct arithmetic inputs to `trend_pct` and therefore to the label, using them is leakage |
| `clicks_last_30d`, `clicks_prev_30d`, `sessions_last_30d`, `sessions_prev_30d` | 30 day comparison windows correlated with the label inputs, excluded to be safe |
| `provider_used`, `model_used` | operational / product flags, not search performance signals |

**Features (safe to use, knowable before the label is computed)**

All remaining columns: keyword context (`search_volume`, `competition`, `competition_level`, `cpc`, `content_type`, `main_intent`), content properties (`word_count`, `char_count`, `content_age_days`, `days_since_last_update`), 90 day activity totals (`impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`, `days_with_impressions`, `days_with_sessions`), derived rates (`ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`), and tier / bucket columns (`age_tier`, `age_tier_order`, `freshness_tier`, `word_count_tier`, `char_count_tier`, `impression_tier`, `position_tier`).

In [2]:
# Sort every column into exactly one bucket and prove the buckets cover the whole table.
LABEL_SOURCES = ["trend_direction", "trend_pct"]
CONTEXT       = ["content_id", "client_id"]
EXCLUDED      = [
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d",      "clicks_prev_30d",
    "sessions_last_30d",    "sessions_prev_30d",
    "provider_used",        "model_used",
]

classified = [c for c in LABEL_SOURCES + CONTEXT + EXCLUDED if c in df.columns]
FEATURES   = [c for c in df.columns if c not in classified]

print(f"label sources : {[c for c in LABEL_SOURCES if c in df.columns]}")
print(f"context       : {CONTEXT}")
print(f"excluded      : {EXCLUDED}")
print(f"features      : {len(FEATURES)} columns")

# Every raw column should land in exactly one bucket.
covered = set(FEATURES) | set(classified)
gap     = set(df.columns) - covered
overlap = [c for c in FEATURES if c in classified]
print(f"\ngap (unclassified) : {gap}   overlap : {overlap}")
print("partition holds" if not gap and not overlap else "CHECK FAILED")


label sources : ['trend_direction', 'trend_pct']
context       : ['content_id', 'client_id']
excluded      : ['impressions_last_30d', 'impressions_prev_30d', 'clicks_last_30d', 'clicks_prev_30d', 'sessions_last_30d', 'sessions_prev_30d', 'provider_used', 'model_used']
features      : 32 columns

gap (unclassified) : set()   overlap : []
partition holds


## 3. Verify it with queries (grain, counts, missing values, windows)

Every claim in section 1 and 2 gets a query cell here. Numbers have to match what the sections say, not just look plausible.

In [3]:
# --- grain check ---
print("=== grain ===")
dupes = df["content_id"].duplicated().sum()
print(f"duplicate content_id rows : {dupes}   (0 = one row per page)")

# --- row and client counts ---
print("\n=== counts ===")
print(f"total rows    : {len(df):,}")
print(f"total clients : {df['client_id'].nunique()}")

# --- label distribution ---
label = (df["trend_direction"] == "down").astype(int)
print(f"\n=== label (proxy) ===")
print(f"positive rate (declining) : {label.mean():.3f}  ({label.sum():,} of {len(df):,})")

# --- missingness per feature, grouped by content_type ---
print("\n=== missingness by content_type (top columns) ===")
miss_cols = ["search_volume", "competition", "cpc", "word_count", "avg_position"]
for col in miss_cols:
    overall = df[col].isna().mean()
    by_type = df.groupby("content_type")[col].apply(lambda s: s.isna().mean())
    print(f"\n  {col}  overall missing: {overall:.2%}")
    print(by_type.to_string())

# --- avg_position = 0 means no data ---
print(f"\n=== avg_position == 0 (no position data, not rank zero) ===")
print(f"rows where avg_position == 0 : {(df['avg_position'] == 0).sum():,}")

# --- rate columns sanity check (should be small percentages, not 0-1 fractions) ---
print("\n=== rate column ranges (all x100 percentages) ===")
for col in ["ctr", "engagement_rate", "scroll_rate", "ai_traffic_pct"]:
    s = df[col].dropna()
    print(f"  {col:20s}  min={s.min():.2f}  median={s.median():.2f}  max={s.max():.2f}")


=== grain ===
duplicate content_id rows : 0   (0 = one row per page)

=== counts ===
total rows    : 30,000
total clients : 32

=== label (proxy) ===
positive rate (declining) : 0.542  (16,262 of 30,000)

=== missingness by content_type (top columns) ===

  search_volume  overall missing: 8.23%
content_type
comparison article    0.000000
feedly article        1.000000
keyword article       0.013673

  competition  overall missing: 8.23%
content_type
comparison article    0.000000
feedly article        1.000000
keyword article       0.013673

  cpc  overall missing: 8.23%
content_type
comparison article    0.000000
feedly article        1.000000
keyword article       0.013673

  word_count  overall missing: 25.66%
content_type
comparison article    0.000000
feedly article        0.000000
keyword article       0.282979

  avg_position  overall missing: 0.00%
content_type
comparison article    0.0
feedly article        0.0
keyword article       0.0

=== avg_position == 0 (no position data

## 4. Data limits

Three things this data genuinely cannot tell you, stated plainly:

**The label is a current window bucket, not a future outcome.** `is_declining_label` is 1 when impressions dropped over the last 30 days compared to the 30 days before that. That is a snapshot of what already happened, not a prediction of what will happen next month. A page flagged as declining right now might already be recovering. Treating this proxy as ground truth is fine for learning the workflow on the starter slice, but it is worth being honest about in any write up.

**Missingness follows content type, not randomness.** Keyword columns (`search_volume`, `competition`, `cpc`) are blank for essentially all `feedly article` rows and present for most `keyword article` rows. Word count is missing for about 26% of rows across types. A blind fill with zero would quietly encode content type into those features. The fix is to add indicator flags (`has_keyword_data`, `has_word_count`) alongside any numeric fill, which the prep script already does for some columns.

**`avg_position = 0` is not rank zero.** There are 1,205 rows where `avg_position` is 0, meaning Google Search Console returned no position data for that page in the window, not that the page ranked first. Using 0 as a numeric feature would tell the model those pages are the best ranked pages in the dataset, which is the opposite of the truth. Replace 0 with null before any modeling step, or use the `position_tier` column which already handles this correctly.

In [4]:
# Back up the three limits with numbers from the data.

# 1. Label is a current window bucket, not a future outcome.
print("=== label proxy check ===")
print(df["trend_direction"].value_counts().to_string())
print(f"\nrows where trend_pct is blank (prev window was zero): {df['trend_pct'].isna().sum():,}")
print("-> the label is a current snapshot; pages marked 'down' may already be recovering")

# 2. Missingness follows content type, not randomness.
print("\n=== keyword missingness by content_type ===")
print(df.groupby("content_type")["search_volume"].apply(lambda s: s.isna().mean()).round(3).to_string())
print("\n=== word_count missingness by content_type ===")
print(df.groupby("content_type")["word_count"].apply(lambda s: s.isna().mean()).round(3).to_string())
print("-> blind fillna(0) would encode content type silently; use indicator flags instead")

# 3. avg_position == 0 means no data, not rank zero.
print(f"\n=== avg_position == 0 rows ===")
zero_pos = df[df["avg_position"] == 0]
print(f"count : {len(zero_pos):,}")
print(f"their median impressions_90d : {zero_pos['impressions_90d'].median():.0f}")
print("-> these are low visibility pages with no GSC position returned, not top ranked pages")


=== label proxy check ===
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152

rows where trend_pct is blank (prev window was zero): 3,388
-> the label is a current snapshot; pages marked 'down' may already be recovering

=== keyword missingness by content_type ===
content_type
comparison article    0.000
feedly article        1.000
keyword article       0.014

=== word_count missingness by content_type ===
content_type
comparison article    0.000
feedly article        0.000
keyword article       0.283
-> blind fillna(0) would encode content type silently; use indicator flags instead

=== avg_position == 0 rows ===
count : 1,205
their median impressions_90d : 1
-> these are low visibility pages with no GSC position returned, not top ranked pages


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled: markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`, then submit your repo URL on the card. Done.